In [23]:
import requests

# ── Config ─────────────────────────────────────────────────────────────────
CTM_LIVE = "https://carbontransitionmodel.com/api/"
CTM_BETA = "https://beta.carbontransitionmodel.com/api/"

HEADERS = {
    "Content-Type": "application/json",
    "Model": "CTM",
}

# Clusters as named in the API
CLUSTERS = [
    "noord_nederland",
    "nzkg",
    "rotterdam_moerdijk",
    "zeeland_west_brabant",
    "chemelot",
    "cluster_6",
]

SECTORS = [
    "other_chemicals",
    "aluminium",
    "other_metals",
    "non_metallic_minerals",
    "transport_equipment",
    "machinery",
    "mining_and_quarrying",
    "food",
    "paper",
    "central_ict",
    "wood_and_wood_products",
    "construction",
    "textile_and_leather",
    "other",
]


FIELDS = [
    "enabled",
    "transformation",
    "electricity_demand",
    "natural_gas_demand",
    "hydrogen_(>98%_vol%)demand",
    "hydrogen(<98%_vol%)demand",
    "heat(<_100_c)demand",
    "heat(>_100_c)demand",
    "waste_gases_demand",
    "coal_and_coal_products_demand",
    "oil_and_oil_products_demand",
    "biomass(solid)demand",
    "biomass(liquid)demand",
    "green_gas_demand",
    "waste(fossil)demand",
    "waste(biogenic)demand",
    "ammonia_demand",
    "methanol_demand",
    "electricity_production",
    "natural_gas_production",
    "hydrogen(>98%_vol%)production",
    "hydrogen(<98%_vol%)production",
    "heat(<_100_c)production",
    "heat(>_100_c)production",
    "waste_gases_production",
    "coal_and_coal_products_production",
    "oil_and_oil_products_production",
    "biomass(solid)production",
    "biomass(liquid)production",
    "green_gas_production",
    "waste(fossil)production",
    "waste(biogenic)_production",
    "ammonia_production",
    "methanol_production",
    "co2_emissions_production",
    "methane_emissions_production",
    "n2o_emissions_production",
    "f_gas_emissions_production",
    "other_greenhouse_gas_emissions_production",
    "co2_fossil_demand",
    "co2_bio_demand",
    "co2_fossil_production",
    "co2_bio_production",
    "hybrid_heat_capacity",
    "chp_capacity",
    "chp_thermal_efficiency",
    "chp_electrical_efficiency",
    "hybrid_heat_flh",
    "chp_flh",
]



# ── Core API call ──────────────────────────────────────────────────────────

class CTMClient:
    def __init__(self, use_beta: bool = False):
        self.url            = CTM_BETA if use_beta else CTM_LIVE
        self.session_id     = None
        self.ms_graph_session = None

    def _call(self, payload: dict) -> dict:
        """Make a raw API call, reusing MSGraphSession for speed."""
        if self.ms_graph_session:
            payload["MSGraphSession"] = self.ms_graph_session
        response = requests.post(self.url, headers=HEADERS, json=payload)
        response.raise_for_status()
        data = response.json()
        # always update the graph session token
        self.ms_graph_session = data.get("MSGraphSession", self.ms_graph_session)
        if "warnings" in data:
            print(f"  [CTM warn] {data['warnings']}")
        return data

    # ── Session management ─────────────────────────────────────────────────

    def create_clean_sheet_session(self) -> str:
        """
        Create a session from base and immediately apply clean sheet settings.
        This disables all CTM built-in calculations so only explicitly set data
        is sent to the ETM.
        """
        # Step 1: create session from base
        data = self._call({"ScenarioID": "base", "outputs": []})
        self.session_id = data["SessionID"]
        print(f"Session created: {self.session_id}")

        # Step 2: apply clean sheet settings (disable all built-in calculations)
        self.set_inputs({
            "other_settings_other_industry_disable_inputs_input": "1",
            "other_settings_ctm_bottom_up_sites_input":           "1",
            "other_settings_disable_waste_incineration_to_etm_input": "1",
            "other_settings_fertilizers_to_chemicals_etm_input": "1",
        })
        print("Clean sheet applied.")
        return self.session_id

    def delete_session(self):
        """Permanently delete the current session."""
        if not self.session_id:
            return
        # deleteSession returns plain text, not JSON
        response = requests.post(
            self.url,
            headers=HEADERS,
            json={"SessionID": self.session_id, "special": ["deleteSession"]},
        )
        print(f"Session deleted: {response.text}")
        self.session_id     = None
        self.ms_graph_session = None

    # ── Inputs ─────────────────────────────────────────────────────────────

    def set_inputs(self, inputs: dict, outputs: list = None) -> dict:
        """Set one or more input values. Returns output values if requested."""
        data = self._call({
            "SessionID": self.session_id,
            "inputs":    inputs,
            "outputs":   outputs or [],
        })
        return data.get("output_values", {})

    # ── Data level helpers ─────────────────────────────────────────────────

    def set_sector(self, sector: str, column: str, value: str,
                   disable_clusters: bool = True):
        """
        Set a value at sector level and optionally disable all clusters
        to avoid ambiguity (recommended per docs).
        """
        inputs = {
            f"{sector}&&sector&&enabled":    "1",
            f"{sector}&&sector&&{column}":   value,
        }
        if disable_clusters:
            for cluster in CLUSTERS:
                inputs[f"{sector}&&{cluster}&&cluster&&enabled"] = "0"
        self.set_inputs(inputs)

    def set_cluster(self, sector: str, cluster: str, column: str, value: str,
                    disable_sector: bool = True):
        """
        Set a value at cluster level and optionally disable the sector level.
        """
        inputs = {
            f"{sector}&&{cluster}&&cluster&&enabled": "1",
            f"{sector}&&{cluster}&&cluster&&{column}": value,
        }
        if disable_sector:
            inputs[f"{sector}&&sector&&enabled"] = "0"
        self.set_inputs(inputs)

    def set_site(self, sector: str, cluster: str, site: str,
                 column_values: dict,
                 disable_sector: bool = True,
                 disable_cluster: bool = True):
        """
        Enable a site and set one or more column values for it.
        Optionally disable sector and cluster levels to avoid ambiguity.

        column_values: dict of {column: value}, e.g.
            {"electricity_demand": "100", "natural_gas_demand": "200"}
        """
        inputs = {
            f"{sector}&&{cluster}&&{site}&&enabled": "1",
        }
        for column, value in column_values.items():
            inputs[f"{sector}&&{cluster}&&{site}&&{column}"] = value

        if disable_sector:
            inputs[f"{sector}&&sector&&enabled"] = "0"
        if disable_cluster:
            inputs[f"{sector}&&{cluster}&&cluster&&enabled"] = "0"

        self.set_inputs(inputs)

    def set_bottom_up_site(self, site: str, column_values: dict):
        """
        Set values for a bottom-up site (Shell Pernis, Tata Steel, etc.).
        These use a flat sitename&&column pattern with no sector/cluster prefix.
        """
        inputs = {f"{site}&&enabled": "1"}
        for column, value in column_values.items():
            inputs[f"{site}&&{column}"] = value
        self.set_inputs(inputs)

    # ── Outputs ────────────────────────────────────────────────────────────

    def get_outputs(self, output_ids: list) -> dict:
        """Read one or more output values."""
        data = self._call({
            "SessionID": self.session_id,
            "outputs":   output_ids,
        })
        return data.get("output_values", {})

    def get_all_outputs(self) -> dict:
        """Dump all available output values."""
        data = self._call({
            "SessionID": self.session_id,
            "outputs":   [],
            "special":   ["requestOutputList"],
        })
        return data.get("output_values", {})

    # ── ETM coupling (for later) ───────────────────────────────────────────

    def couple_etm(self, etm_scenario_id: str = None, etm_session_id: str = None,
                   auth_token: str = None):
        """
        Couple the CTM session to an ETM scenario.
        Provide either etm_scenario_id (saved) or etm_session_id (live).
        """
        inputs = {"etm_coupling_switch": "1"}
        if etm_scenario_id:
            inputs["etm_scenario_id"] = etm_scenario_id
        if etm_session_id:
            inputs["etm_session_id"] = etm_session_id
        if auth_token:
            inputs["etm_authorization_token"] = auth_token

        outputs = ["etm_session_id"] if etm_scenario_id else []
        result = self.set_inputs(inputs, outputs=outputs)
        if "etm_session_id" in result:
            print(f"ETM session: {result['etm_session_id']}")
        return result

    def get_available_sites(self) -> dict:
        """
        Returns all sites known to the CTM, grouped by category:
        - 'sector_cluster': regular sites under sector/cluster hierarchy
        - 'bottom_up': large industrial sites (Shell Pernis, Tata Steel etc.)
        - 'custom': ##new_cc_siteN## placeholder slots
        """

        print(f'sesh id: {self.session_id}')
        data = self._call({
            "SessionID": self.session_id,
            "outputs":   [],
            "special":   ["requestFullList"],
        })
        all_keys = data.get("output_values", {}).keys()

        sector_cluster, bottom_up, custom = set(), set(), set()

        for key in all_keys:
            if not key.endswith("&&enabled"):
                continue
            parts = key.split("&&")
            if key.startswith("##new_cc_site"):
                custom.add(parts[0])
            elif len(parts) == 2:
                # sitename&&enabled -> bottom-up
                bottom_up.add(parts[0])
            elif len(parts) == 4:
                # sector&&cluster&&site&&enabled -> regular site
                sector_cluster.add(parts[2])

        return {
            "sector_cluster": sorted(sector_cluster),
            "bottom_up":      sorted(bottom_up),
            "custom":         sorted(custom),
        }
    
    

In [2]:
from ctm_client import CTMClient

ctm = CTMClient(use_beta=True)  # switch to True for testing

# ── 1. Create a clean sheet session ───────────────────────────────────────
ctm.create_clean_sheet_session()

try:
    # ── 2a. Sector level ──────────────────────────────────────────────────
    ctm.set_sector(
        sector  = "other_chemicals",
        column  = "natural_gas_demand",
        value   = "3000",
    )

    # ── 2b. Cluster level ─────────────────────────────────────────────────
    ctm.set_cluster(
        sector  = "food",
        cluster = "cluster_6",
        column  = "electricity_demand",
        value   = "500",
    )

    # ── 2c. Regular site level ────────────────────────────────────────────
    ctm.set_site(
        sector  = "food",
        cluster = "cluster_6",
        site    = "heineken_nederland_brouwerij_zoeterwoude",
        column_values = {
            "electricity_demand":  "100",
            "natural_gas_demand":  "200",
        },
    )

    # ── 2d. Bottom-up site ────────────────────────────────────────────────
    ctm.set_bottom_up_site(
        site = "shell_pernis",
        column_values = {
            "electricity_demand":  "2000",
            "natural_gas_demand":  "8000",
        },
    )

    # ── 3. Read outputs ───────────────────────────────────────────────────
    # outputs = ctm.get_outputs(["output_ID_1", "output_ID_2"])
    # print(outputs)
    # outputs = ctm.get_all_outputs()
    # print(list(outputs.keys())[:20])

    # ── 4. Couple to ETM (when ready) ─────────────────────────────────────
    # ctm.couple_etm(etm_scenario_id="1234")


    sites = ctm.get_available_sites()

    print(f"Regular sites: {sites['sector_cluster']}")
    print(f"Bottom-up sites: {len(sites['bottom_up'])}")
    print(f"Custom slots available: {len(sites['custom'])}")

    # Search for a specific plant
    search = "heineken"
    matches = [s for s in sites['sector_cluster'] if search in s.lower()]
    print(f"\nMatches for '{search}': {matches}")

    # print(f'Regular sites: {sites['sector_cluster']}')

finally:
    # always clean up
    # ctm.delete_session()
    print('x')

Session created: SE-ac4d2f07e02f8aec
Clean sheet applied.
sesh id: SE-ac4d2f07e02f8aec
Regular sites: ['a12_cpp_petrogas_ep_netherlands', 'aardgasbuffer_zuidwending', 'abbott_healthcare_products', 'abbott_laboratories', 'adm_europoort', 'agc_flat_glass_nederland', 'agristo', 'alco_energy_rotterdam', 'aluminium_chemie_rotterdam', 'ams05_ams07', 'ams10', 'ams15_de_president', 'ams5', 'ams6', 'ams7', 'ams8', 'amsteldijk_beheer', 'apollo_vredestein', 'ardagh_glass_dongen', 'ardagh_glass_moerdijk', 'asfalt_centrale_bam', 'asfalt_centrale_nijkerk', 'asfalt_centrale_rotterdam', 'asfalt_centrale_utrecht', 'asfalt_productie_amsterdam', 'asfalt_productie_de_eem', 'asfalt_productie_doetinchem', 'asfalt_productie_hoogblokland', 'asfalt_productie_nijmegen', 'asfalt_productie_rasenberg_infra', 'asfalt_productie_rotterdam_rijnmond', 'asfalt_productie_tiel', 'asfalt_productie_westerbroek', 'asfalt_produktie_maatschappij', 'asfaltcentrale_harderwijk_bruil', 'asfaltcentrale_harderwijk_kws', 'asfaltcentr

In [3]:
print(ctm.session_id)

SE-ac4d2f07e02f8aec


In [22]:
# dump everything for one site and see what fields exist
all_data = ctm._call({
    "SessionID": ctm.session_id,
    "outputs": [],
    "special": ["requestFullList"],
}).get("output_values", {})

# pick any known site and see all its keys
site = "cargill_bergen"
site_keys = {k: v for k, v in all_data.items() if site in k}
print(site_keys)

HTTPError: 500 Server Error: INTERNAL SERVER ERROR for url: https://beta.carbontransitionmodel.com/api/

In [5]:
import polars as pl
df = pl.DataFrame(
    {
        "key": list(site_keys.keys()),
        "value": list(site_keys.values()),
    },
    strict=False
)

df

key,value
str,str
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
…,…
"""site_shell_moerdijk_waste_gas_…","""n/a"""
"""site_shell_moerdijk_waste_gas_…","""0"""
"""site_shell_moerdijk_waste_gas_…","""n/a"""


In [6]:
aux = df.filter(pl.col('key').str.contains('gas'))
aux

key,value
str,str
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
…,…
"""site_shell_moerdijk_waste_gas_…","""n/a"""
"""site_shell_moerdijk_waste_gas_…","""0"""
"""site_shell_moerdijk_waste_gas_…","""n/a"""


In [7]:
auxx = aux.filter(pl.col('key').str.contains('heat'))
auxx

key,value
str,str
"""shell_moerdijk_heat_integratio…","""242.3"""
"""shell_moerdijk_heat_integratio…","""1"""
"""shell_moerdijk_heat_integratio…","""0.7"""
"""site_shell_moerdijk_etm_heat_g…","""n/a"""
"""site_shell_moerdijk_heat_gasbo…","""n/a"""
"""site_shell_moerdijk_heat_gasbo…","""n/a"""


In [8]:
aux2 = df.with_columns(
    pl.col("key")
      .str.split("&&")
      .list.get(0)
      .alias("sector")
).filter(pl.col('sector').is_in(SECTORS)).with_columns(pl.col("key")
      .str.split("&&")
      .list.get(1)
      .alias("cluster")).filter(pl.col('cluster').is_in(CLUSTERS))
aux2

key,value,sector,cluster
str,str,str,str


In [9]:
aux3 = aux2.with_columns(
    pl.col("key")
      .str.split("&&")
      .list.get(2)
      .alias("plant")
).filter((~pl.col('plant').is_in(FIELDS)) & (~pl.col('plant').str.contains('cluster')) & (~pl.col('plant').str.contains('residue')))
plantsss = aux3.select('plant').unique()
plantsss

plant
str


In [10]:
import polars as pl
plants = pl.read_csv('/home/307920@ontw.alfa.local/projects/epn-ma-master/data/dsh/20260508_new/datasafehouse-plant-export.csv.csv')

In [11]:
plants = plants.with_columns(pl.col("Plant name")
    .str.to_lowercase()
    .str.replace_all(" ", "_")
    .alias("Plant name updated"))

plants

Latitude,Longitude,Address,Zip code,City,SBI code(s),Description,Plant name,Plant identifier,Existing plant/New plant,KvK number(s),All EANs,Plant name updated
f64,f64,str,str,str,str,str,str,str,str,i64,str,str
51.956512,4.122673,"""Elbeweg 125""","""3198LC""","""Europoort""","""1089 - Vervaardiging van overi…",null,"""ADM Europoort""","""527be9a2-5258-4cda-89cb-b48520…","""existing""",null,"""[{""ean_code"":""8716892760000783…","""adm_europoort"""
52.40095,4.792595,"""australiehavenweg 21""","""1045BA""","""Amsterdam""","""35111 - Productie van elektric…",null,"""AEB - AEC / HRC""","""0ac0ee36-995e-4723-a75a-4ce872…","""existing""",58867937,"""[{""ean_code"":""8716859000000026…","""aeb_-_aec_/_hrc"""
52.405947,4.85313,"""Petroleumhavenweg 1B""","""1041AB""","""Amsterdam""","""35111 - Productie van elektric…",null,"""AEB - BEC""","""dd749763-a050-43b8-96f3-7e1e58…","""existing""",58867937,"""[{""ean_code"":""8716859200033329…","""aeb_-_bec"""
51.50933,4.255797,"""Plasticslaan 1""","""4600AC""","""Bergen op Zoom""","""2011 - Vervaardiging van indus…",null,"""Air Liquide Bergen op Zoom""","""48fb5231-1d91-4906-97e7-8c4433…","""existing""",null,"""[{""ean_code"":""8717185180030155…","""air_liquide_bergen_op_zoom"""
null,null,"""Keteldiep 1C""","""4791SM""","""Moerdijk - Klundert""","""2011 - Vervaardiging van indus…",null,"""Air Liquide Booster Moerdijk""","""00f1a6e2-a623-4c38-ad43-ef9be2…","""existing""",null,"""[{""ean_code"":""8716879100004675…","""air_liquide_booster_moerdijk"""
…,…,…,…,…,…,…,…,…,…,…,…,…
51.910616,4.217811,"""Neckarweg 5""","""3198LJ""","""Europoort-Rotterdam""","""1920 - Aardolieverwerking""",null,"""VPR Europoort""","""b08ff0e9-5b03-448e-8b57-aa2b9e…","""existing""",33298434,"""[{""ean_code"":""8716892760000022…","""vpr_europoort"""
52.461595,4.586851,"""Halkade 4""","""1976DC""","""IJmuiden""","""4321 - Elektrotechnische bouwi…",null,"""Walstroom Halkade""","""3d6d705e-5b33-4517-a59d-8a9152…","""new""",34065340,"""[{""ean_code"":null,""grid_operat…","""walstroom_halkade"""
52.459608,4.569951,"""Monnickendamkade""","""1976DC""","""IJmuiden""","""4321 - Elektrotechnische bouwi…",null,"""Walstroom Monnickendamkade""","""1212af82-ffdc-4ed1-95c2-11c1ad…","""new""",34065340,"""[{""ean_code"":null,""grid_operat…","""walstroom_monnickendamkade"""


In [12]:
aux3

key,value,sector,cluster,plant
str,str,str,str,str


In [13]:

fields_split = {
    'Plant name': 'location',
    'Address':    'adress',     #food&&zeeland_west_brabant&&cargill_bergen_op_zoom&&adress
    'ZIP':        'zip_code',
    'City':       'city',
}

fields_suffix = {
    'Latitude':  'latitude_dashboard',  #site_cargill_bergen_op_zoom_latitude_dashboard
    'Longitude': 'longitude_dashboard',
}

another = (
    aux3
    .with_columns(field=pl.col("key").str.split("&&").list.get(-1))
    .with_columns(
        *[
            pl.when(pl.col("field") == field)
            .then(pl.col("value"))
            .alias(col)
            for col, field in fields_split.items()
        ],
        *[
            pl.when(pl.col("key").str.ends_with(suffix))
            .then(pl.col("value"))
            .alias(col)
            for col, suffix in fields_suffix.items()
        ],
    )
)
another


key,value,sector,cluster,plant,field,Plant name,Address,ZIP,City,Latitude,Longitude
str,str,str,str,str,str,str,str,str,str,str,str


In [14]:
fin = (
    another
    .group_by(["sector", "cluster", "plant"])
    .agg(
        pl.col("Plant name").drop_nulls().first(),
        pl.col("Address").drop_nulls().first(),
        pl.col("Latitude").drop_nulls().first(),
        pl.col("Longitude").drop_nulls().first(),
        pl.col("ZIP").drop_nulls().first(),
        pl.col("City").drop_nulls().first(),
    )
)

fin

sector,cluster,plant,Plant name,Address,Latitude,Longitude,ZIP,City
str,str,str,str,str,str,str,str,str


In [15]:
aux3

key,value,sector,cluster,plant
str,str,str,str,str


In [16]:
aux4 = aux3.select(['sector', 'cluster', 'plant']).unique()
aux4

sector,cluster,plant
str,str,str


In [17]:
joined =aux4.join(plants,
              left_on='plant',
              right_on='Plant name updated',
              how='inner')

joined

sector,cluster,plant,Latitude,Longitude,Address,Zip code,City,SBI code(s),Description,Plant name,Plant identifier,Existing plant/New plant,KvK number(s),All EANs
str,str,str,f64,f64,str,str,str,str,str,str,str,str,i64,str


In [18]:
plants

Latitude,Longitude,Address,Zip code,City,SBI code(s),Description,Plant name,Plant identifier,Existing plant/New plant,KvK number(s),All EANs,Plant name updated
f64,f64,str,str,str,str,str,str,str,str,i64,str,str
51.956512,4.122673,"""Elbeweg 125""","""3198LC""","""Europoort""","""1089 - Vervaardiging van overi…",null,"""ADM Europoort""","""527be9a2-5258-4cda-89cb-b48520…","""existing""",null,"""[{""ean_code"":""8716892760000783…","""adm_europoort"""
52.40095,4.792595,"""australiehavenweg 21""","""1045BA""","""Amsterdam""","""35111 - Productie van elektric…",null,"""AEB - AEC / HRC""","""0ac0ee36-995e-4723-a75a-4ce872…","""existing""",58867937,"""[{""ean_code"":""8716859000000026…","""aeb_-_aec_/_hrc"""
52.405947,4.85313,"""Petroleumhavenweg 1B""","""1041AB""","""Amsterdam""","""35111 - Productie van elektric…",null,"""AEB - BEC""","""dd749763-a050-43b8-96f3-7e1e58…","""existing""",58867937,"""[{""ean_code"":""8716859200033329…","""aeb_-_bec"""
51.50933,4.255797,"""Plasticslaan 1""","""4600AC""","""Bergen op Zoom""","""2011 - Vervaardiging van indus…",null,"""Air Liquide Bergen op Zoom""","""48fb5231-1d91-4906-97e7-8c4433…","""existing""",null,"""[{""ean_code"":""8717185180030155…","""air_liquide_bergen_op_zoom"""
null,null,"""Keteldiep 1C""","""4791SM""","""Moerdijk - Klundert""","""2011 - Vervaardiging van indus…",null,"""Air Liquide Booster Moerdijk""","""00f1a6e2-a623-4c38-ad43-ef9be2…","""existing""",null,"""[{""ean_code"":""8716879100004675…","""air_liquide_booster_moerdijk"""
…,…,…,…,…,…,…,…,…,…,…,…,…
51.910616,4.217811,"""Neckarweg 5""","""3198LJ""","""Europoort-Rotterdam""","""1920 - Aardolieverwerking""",null,"""VPR Europoort""","""b08ff0e9-5b03-448e-8b57-aa2b9e…","""existing""",33298434,"""[{""ean_code"":""8716892760000022…","""vpr_europoort"""
52.461595,4.586851,"""Halkade 4""","""1976DC""","""IJmuiden""","""4321 - Elektrotechnische bouwi…",null,"""Walstroom Halkade""","""3d6d705e-5b33-4517-a59d-8a9152…","""new""",34065340,"""[{""ean_code"":null,""grid_operat…","""walstroom_halkade"""
52.459608,4.569951,"""Monnickendamkade""","""1976DC""","""IJmuiden""","""4321 - Elektrotechnische bouwi…",null,"""Walstroom Monnickendamkade""","""1212af82-ffdc-4ed1-95c2-11c1ad…","""new""",34065340,"""[{""ean_code"":null,""grid_operat…","""walstroom_monnickendamkade"""


In [19]:
joined2 =fin.join(plants.select(['Latitude', 'Longitude', 'Address', 'Zip code', 'City', 'Plant name']),
              left_on='Plant name',
              right_on='Plant name',
              how='inner')

joined2

sector,cluster,plant,Plant name,Address,Latitude,Longitude,ZIP,City,Latitude_right,Longitude_right,Address_right,Zip code,City_right
str,str,str,str,str,str,str,str,str,f64,f64,str,str,str


### What?


In [20]:
df.filter(~pl.col('key').str.contains('new_cc_site'))

key,value
str,str
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
"""ldsh&&other_chemicals&&rotterd…",""""""
…,…
"""site_shell_moerdijk_waste_gas_…","""n/a"""
"""site_shell_moerdijk_waste_gas_…","""0"""
"""site_shell_moerdijk_waste_gas_…","""n/a"""


In [21]:
# ctm.delete_session()